# Multimodal CrisisMMD Model Testing
This notebook builds a multimodal classifier to improve F1 score by combining:
- OpenAI API text embeddings from `tweet_text`
- DenseNet image embeddings from disaster images
- An attention-based fusion head for final classification

## Notes before running
1. Set your API key in the environment: `OPENAI_API_KEY`
2. Start with a small sample for quick tests, then scale up
3. This pipeline reports weighted F1, macro F1, and accuracy on validation/test splits

In [3]:

# Setup, imports, and tuning variables
import os
import pickle
import random
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import DenseNet121_Weights

from openai import OpenAI

# -------------------------------
# Tunable experiment configuration
# -------------------------------
SEED = 42

# Data/task options
TARGET_COLUMN = 'text_human'        # try: 'image_human', 'image_info'
TEXT_COLUMN = 'tweet_text'
IMAGE_REL_PATH_COLUMN = 'image_path'
DISASTER_COLUMN = 'disaster_type'
USE_SMALL_SAMPLE = True
SMALL_SAMPLE_SIZE = 1000
TRAIN_HOLDOUT_RATIO = 0.30          # 70% train, 30% holdout
VAL_FROM_HOLDOUT_RATIO = 0.50       # 15% val, 15% test

# Text embedding mode
USE_OPENAI = False                  # False = zero-token local dry run
OPENAI_EMBED_MODEL = 'text-embedding-3-small'
LOCAL_TEXT_EMBED_DIM = 1536         # keep close to OpenAI embedding size

# Model/training knobs
BATCH_SIZE = 16
NUM_WORKERS = 0                     # 0 = main process only (required for Python 3.14+ in notebooks)
EPOCHS = 2
LR = 1e-4
WEIGHT_DECAY = 1e-4
D_MODEL = 512
N_HEADS = 8
DROPOUT = 0.2
FREEZE_IMAGE_BACKBONE = False
EARLY_STOPPING_PATIENCE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

def find_project_root(marker_dir='CrisisMMD_v2.0') -> Path:
    cwd = Path.cwd().resolve()
    home = Path.home().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (candidate / marker_dir).is_dir():
            return candidate

    for root in [home / 'Projects', home / 'Desktop', home]:
        if not root.exists():
            continue
        direct = root / marker_dir
        if direct.is_dir():
            return root
        hit = next(root.rglob(marker_dir), None)
        if hit and hit.is_dir():
            return hit.parent

    raise FileNotFoundError(f"Could not find '{marker_dir}' from cwd={cwd}")

PROJECT_ROOT = find_project_root()
CRISIS_ROOT = PROJECT_ROOT / 'CrisisMMD_v2.0'
ANNOTATIONS_DIR = CRISIS_ROOT / 'annotations'
CACHE_DIR = PROJECT_ROOT / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Annotations directory exists:', ANNOTATIONS_DIR.exists())
print('USE_OPENAI:', USE_OPENAI)


Using device: cpu
Project root: /home/Ktang/Projects/DataMiningProject/crisisMMDDataSet
Annotations directory exists: True
USE_OPENAI: False


In [ ]:
# Load CrisisMMD annotations and prepare one multimodal task
all_dataframes = []
for filename in os.listdir(ANNOTATIONS_DIR):
    if filename.endswith('.tsv') and not filename.startswith('._'):
        df_tmp = pd.read_csv(ANNOTATIONS_DIR / filename, sep='\t')
        df_tmp[DISASTER_COLUMN] = filename.replace('_final_data.tsv', '')
        all_dataframes.append(df_tmp)

master_df = pd.concat(all_dataframes, ignore_index=True)
master_df = master_df.dropna(subset=[TARGET_COLUMN, TEXT_COLUMN, IMAGE_REL_PATH_COLUMN]).copy()

# Remove rows whose image file does not exist
master_df['full_image_path'] = master_df[IMAGE_REL_PATH_COLUMN].apply(lambda p: CRISIS_ROOT / p)
master_df = master_df[master_df['full_image_path'].apply(lambda p: p.exists())].reset_index(drop=True)

print('Rows after filtering:', len(master_df))
print('Target distribution (top 20):')
print(master_df[TARGET_COLUMN].value_counts().head(20))

# Optional small-sample mode for fast iteration
if USE_SMALL_SAMPLE and len(master_df) > SMALL_SAMPLE_SIZE:
    master_df, _ = train_test_split(
        master_df,
        train_size=SMALL_SAMPLE_SIZE,
        random_state=SEED,
        stratify=master_df[TARGET_COLUMN]
    )
    master_df = master_df.reset_index(drop=True)
    print('Using sampled rows:', len(master_df))

label_encoder = LabelEncoder()
master_df['label_id'] = label_encoder.fit_transform(master_df[TARGET_COLUMN].astype(str))
NUM_CLASSES = len(label_encoder.classes_)
print('Number of classes:', NUM_CLASSES)

def can_stratify(labels: pd.Series, test_size: float) -> bool:
    counts = labels.value_counts()
    if len(counts) < 2:
        return False
    n_samples = len(labels)
    n_test = int(np.ceil(n_samples * test_size))
    n_train = n_samples - n_test
    return counts.min() >= 2 and n_train >= len(counts) and n_test >= len(counts)

first_stratify = master_df['label_id'] if can_stratify(master_df['label_id'], TRAIN_HOLDOUT_RATIO) else None
if first_stratify is None:
    print('Warning: first split cannot be stratified (rare classes too small). Using random split.')

train_df, temp_df = train_test_split(
    master_df,
    test_size=TRAIN_HOLDOUT_RATIO,
    random_state=SEED,
    stratify=first_stratify
)

second_stratify = temp_df['label_id'] if can_stratify(temp_df['label_id'], VAL_FROM_HOLDOUT_RATIO) else None
if second_stratify is None:
    print('Warning: val/test split cannot be stratified (rare classes too small). Using random split.')

val_df, test_df = train_test_split(
    temp_df,
    test_size=VAL_FROM_HOLDOUT_RATIO,
    random_state=SEED,
    stratify=second_stratify
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('Split sizes:', len(train_df), len(val_df), len(test_df))

Rows after filtering: 18082
Target distribution (top 20):
text_human
other_relevant_information                6499
not_humanitarian                          5227
rescue_volunteering_or_donation_effort    3776
infrastructure_and_utility_damage         1428
injured_or_dead_people                     533
affected_individuals                       517
vehicle_damage                              61
missing_or_found_people                     41
Name: count, dtype: int64
Using sampled rows: 1000
Number of classes: 8


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [ ]:
# Text embeddings: OpenAI mode or no-token local mode
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
client = None

if USE_OPENAI:
    if not OPENAI_API_KEY:
        raise RuntimeError('USE_OPENAI=True but OPENAI_API_KEY is not set.')
    client = OpenAI(api_key=OPENAI_API_KEY)

EMBEDDING_CACHE_PATH = CACHE_DIR / f'openai_embeddings_{OPENAI_EMBED_MODEL}.pkl'
if EMBEDDING_CACHE_PATH.exists():
    with open(EMBEDDING_CACHE_PATH, 'rb') as f:
        text_embedding_cache: Dict[str, np.ndarray] = pickle.load(f)
else:
    text_embedding_cache = {}

print('Cached OpenAI embeddings:', len(text_embedding_cache))

def get_text_embeddings_openai(texts: List[str], batch_size: int = 128) -> np.ndarray:
    rows = [str(t) for t in pd.Series(texts).fillna('').tolist()]
    missing = [t for t in set(rows) if t not in text_embedding_cache]

    if missing:
        for i in tqdm(range(0, len(missing), batch_size), desc='Embedding text with OpenAI'):
            batch = missing[i:i + batch_size]
            response = client.embeddings.create(model=OPENAI_EMBED_MODEL, input=batch)
            for txt, emb in zip(batch, response.data):
                text_embedding_cache[txt] = np.array(emb.embedding, dtype=np.float32)

        with open(EMBEDDING_CACHE_PATH, 'wb') as f:
            pickle.dump(text_embedding_cache, f)

    return np.stack([text_embedding_cache[t] for t in rows]).astype(np.float32)

def get_text_embeddings_local(texts: List[str], n_features: int = LOCAL_TEXT_EMBED_DIM) -> np.ndarray:
    rows = pd.Series(texts).fillna('').astype(str).tolist()
    vec = HashingVectorizer(n_features=n_features, alternate_sign=False, norm=None)
    X = vec.transform(rows)
    return X.astype(np.float32).toarray()

if USE_OPENAI:
    print('Embedding mode: OpenAI')
    X_text_train = get_text_embeddings_openai(train_df[TEXT_COLUMN].tolist())
    X_text_val = get_text_embeddings_openai(val_df[TEXT_COLUMN].tolist())
    X_text_test = get_text_embeddings_openai(test_df[TEXT_COLUMN].tolist())
else:
    print('Embedding mode: LOCAL (no token usage)')
    X_text_train = get_text_embeddings_local(train_df[TEXT_COLUMN].tolist())
    X_text_val = get_text_embeddings_local(val_df[TEXT_COLUMN].tolist())
    X_text_test = get_text_embeddings_local(test_df[TEXT_COLUMN].tolist())

TEXT_DIM = X_text_train.shape[1]
print('Text embedding dim:', TEXT_DIM)

Cached OpenAI embeddings: 0
Embedding mode: LOCAL (no token usage)
Text embedding dim: 1536


In [ ]:
# DenseNet image encoder + attention fusion model
weights = DenseNet121_Weights.IMAGENET1K_V1
image_transform = weights.transforms()

class CrisisMultimodalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, text_embeddings: np.ndarray, transform):
        self.df = df.reset_index(drop=True)
        self.text_embeddings = text_embeddings
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['full_image_path']).convert('RGB')
        image = self.transform(image)
        text_vec = torch.tensor(self.text_embeddings[idx], dtype=torch.float32)
        label = torch.tensor(int(row['label_id']), dtype=torch.long)
        return image, text_vec, label

train_ds = CrisisMultimodalDataset(train_df, X_text_train, image_transform)
val_ds = CrisisMultimodalDataset(val_df, X_text_val, image_transform)
test_ds = CrisisMultimodalDataset(test_df, X_text_test, image_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

class AttentionFusionClassifier(nn.Module):
    def __init__(self, text_dim: int, num_classes: int, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        backbone = models.densenet121(weights=weights)
        self.image_encoder = backbone.features
        self.image_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.image_proj = nn.Linear(1024, d_model)

        if FREEZE_IMAGE_BACKBONE:
            for p in self.image_encoder.parameters():
                p.requires_grad = False

        self.text_proj = nn.Linear(text_dim, d_model)
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, images, text_vecs):
        img_feat_map = self.image_encoder(images)
        img_feat = self.image_pool(img_feat_map).flatten(1)
        img_tok = self.image_proj(img_feat)

        txt_tok = self.text_proj(text_vecs)

        tokens = torch.stack([txt_tok, img_tok], dim=1)
        attn_out, _ = self.attn(tokens, tokens, tokens)
        fused = self.norm(tokens + attn_out).mean(dim=1)

        return self.classifier(fused)

model = AttentionFusionClassifier(
    text_dim=TEXT_DIM,
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    dropout=DROPOUT
).to(DEVICE)

print(model.__class__.__name__)
print(f'Backbone frozen: {FREEZE_IMAGE_BACKBONE}')

AttentionFusionClassifier
Backbone frozen: False


In [ ]:
# Train and evaluate (weighted F1 focus)
class_counts = train_df['label_id'].value_counts().sort_index()
class_weights = torch.tensor(
    [len(train_df) / (NUM_CLASSES * class_counts.get(i, 1)) for i in range(NUM_CLASSES)],
    dtype=torch.float32,
    device=DEVICE
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def run_epoch(loader, train_mode: bool = True):
    model.train(train_mode)
    epoch_loss = 0.0
    y_true, y_pred = [], []

    for images, text_vecs, labels in tqdm(loader, leave=False):
        images = images.to(DEVICE)
        text_vecs = text_vecs.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.set_grad_enabled(train_mode):
            logits = model(images, text_vecs)
            loss = criterion(logits, labels)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        epoch_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        y_true.extend(labels.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())

    avg_loss = epoch_loss / len(loader.dataset)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    return avg_loss, weighted_f1, macro_f1, acc

best_val_f1 = -1.0
best_state = None
epochs_without_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_wf1, train_mf1, train_acc = run_epoch(train_loader, train_mode=True)
    val_loss, val_wf1, val_mf1, val_acc = run_epoch(val_loader, train_mode=False)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss={train_loss:.4f}, wF1={train_wf1:.4f}, mF1={train_mf1:.4f}, acc={train_acc:.4f} | "
        f"Val loss={val_loss:.4f}, wF1={val_wf1:.4f}, mF1={val_mf1:.4f}, acc={val_acc:.4f}"
    )

    if val_wf1 > best_val_f1:
        best_val_f1 = val_wf1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improve = 0
    else:
        epochs_without_improve += 1

    if EARLY_STOPPING_PATIENCE > 0 and epochs_without_improve >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered after {epoch} epochs.")
        break

if best_state is not None:
    model.load_state_dict(best_state)
print('Best validation weighted F1:', round(best_val_f1, 4))
print('Tune knobs: BATCH_SIZE, EPOCHS, LR, WEIGHT_DECAY, D_MODEL, N_HEADS, DROPOUT, FREEZE_IMAGE_BACKBONE, SMALL_SAMPLE_SIZE')

KeyboardInterrupt: 

In [ ]:
# Final test metrics and model save
def predict_loader(loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, text_vecs, labels in tqdm(loader, leave=False):
            images = images.to(DEVICE)
            text_vecs = text_vecs.to(DEVICE)
            logits = model(images, text_vecs)
            preds = logits.argmax(dim=1)
            y_true.extend(labels.numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
    return y_true, y_pred

y_true_test, y_pred_test = predict_loader(test_loader)

test_acc = accuracy_score(y_true_test, y_pred_test)
test_wf1 = f1_score(y_true_test, y_pred_test, average='weighted', zero_division=0)
test_mf1 = f1_score(y_true_test, y_pred_test, average='macro', zero_division=0)

print('Test Accuracy:', round(test_acc, 4))
print('Test Weighted F1:', round(test_wf1, 4))
print('Test Macro F1:', round(test_mf1, 4))

print('\nClassification Report:')
print(classification_report(
    y_true_test,
    y_pred_test,
    target_names=label_encoder.classes_,
    zero_division=0
))

MODEL_PATH = CACHE_DIR / 'multimodal_densenet_attention_openai.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'label_classes': label_encoder.classes_.tolist(),
    'target_column': TARGET_COLUMN,
    'text_model': OPENAI_EMBED_MODEL
}, MODEL_PATH)

print('Saved model to:', MODEL_PATH)

In [ ]:

# Hyperparameter auto-sweep
# Trains a fresh model for each config and reports the best val weighted F1.
# Tip: keep USE_SMALL_SAMPLE=True and EPOCHS small for a fast sweep.

import itertools
import copy

SWEEP_GRID = {
    'LR':                    [1e-4, 3e-4],
    'D_MODEL':               [256, 512],
    'DROPOUT':               [0.2, 0.3],
    'FREEZE_IMAGE_BACKBONE': [False, True],
}

def make_model(text_dim, num_classes, d_model, n_heads, dropout, freeze):
    from torchvision import models
    from torchvision.models import DenseNet121_Weights
    _weights = DenseNet121_Weights.IMAGENET1K_V1
    m = AttentionFusionClassifier(
        text_dim=text_dim,
        num_classes=num_classes,
        d_model=d_model,
        n_heads=n_heads,
        dropout=dropout,
    )
    if freeze:
        for p in m.image_encoder.parameters():
            p.requires_grad = False
    return m.to(DEVICE)

def sweep_train(model_s, lr, patience=EARLY_STOPPING_PATIENCE):
    """Run a quick training loop and return best val weighted F1."""
    cc = train_df['label_id'].value_counts().sort_index()
    cw = torch.tensor(
        [len(train_df) / (NUM_CLASSES * cc.get(i, 1)) for i in range(NUM_CLASSES)],
        dtype=torch.float32, device=DEVICE
    )
    crit = nn.CrossEntropyLoss(weight=cw)
    opt = optim.AdamW(model_s.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    best_f1 = -1.0
    no_improve = 0
    for _ in range(EPOCHS):
        model_s.train()
        for images, text_vecs, labels in train_loader:
            images, text_vecs, labels = images.to(DEVICE), text_vecs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = crit(model_s(images, text_vecs), labels)
            loss.backward()
            opt.step()

        model_s.eval()
        y_true_v, y_pred_v = [], []
        with torch.no_grad():
            for images, text_vecs, labels in val_loader:
                images, text_vecs = images.to(DEVICE), text_vecs.to(DEVICE)
                preds = model_s(images, text_vecs).argmax(1)
                y_true_v.extend(labels.numpy())
                y_pred_v.extend(preds.cpu().numpy())

        val_f1 = f1_score(y_true_v, y_pred_v, average='weighted', zero_division=0)
        if val_f1 > best_f1:
            best_f1 = val_f1
            no_improve = 0
        else:
            no_improve += 1
        if patience > 0 and no_improve >= patience:
            break
    return best_f1

# ---- run the sweep ----
sweep_keys = list(SWEEP_GRID.keys())
sweep_values = list(SWEEP_GRID.values())
sweep_results = []

for combo in itertools.product(*sweep_values):
    cfg = dict(zip(sweep_keys, combo))
    d_model = cfg['D_MODEL']
    # n_heads must divide d_model
    n_heads = 8 if d_model % 8 == 0 else 4

    print(f"Testing: {cfg}  (n_heads={n_heads}) ...", end=' ', flush=True)
    m = make_model(TEXT_DIM, NUM_CLASSES, d_model, n_heads, cfg['DROPOUT'], cfg['FREEZE_IMAGE_BACKBONE'])
    val_f1 = sweep_train(m, cfg['LR'])
    sweep_results.append({**cfg, 'n_heads': n_heads, 'val_wF1': round(val_f1, 4)})
    print(f"val wF1 = {val_f1:.4f}")
    del m
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

sweep_df = pd.DataFrame(sweep_results).sort_values('val_wF1', ascending=False).reset_index(drop=True)
print('\nSweep results (best first):')
print(sweep_df.to_string())
best_cfg = sweep_df.iloc[0].to_dict()
print(f'\nBest config: {best_cfg}')
